## Schritt 1: Setup und Pfade

Wir laden die Bibliotheken und legen fest, welche Dateien gelesen und geschrieben werden.

- `cv2` (OpenCV) liest und schreibt Videos und bearbeitet Bilder.
- `numpy` rechnet mit den Bildern (ein Bild ist ein Zahlen-Array).
- `matplotlib` zeigt Bilder an. Wir nutzen es nur für **bereits pixelierte** Frames.
- `csv`, `time`, `Path` brauchen wir für das Protokoll, die Zeitmessung und die Dateipfade.

Die Pfade sind **relativ zum Projektordner**: `Path.cwd().parent` ist der Ordner über `notebooks/`. So läuft derselbe Code auf dem Laptop und auf dem Klinik-PC, ohne dass du etwas ändern musst.

Die Ordnerstruktur trennt die Daten strikt:
- `data/raw/`: Rohvideo, wird **nie verändert**.
- `data/interim/`: Zwischenergebnisse (hier das Erkennungsprotokoll als CSV).
- `data/processed/`: Endergebnis (das pixelierte Video).

Alle drei Ordner stehen in `.gitignore`, die Videos landen also nicht auf GitHub.

Die Kontrollausgabe am Ende muss zweimal `True` zeigen. Bei `False` stimmt ein Pfad nicht.


In [ ]:
import csv
import sys
import time
from collections import deque
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

print("Python:", sys.executable)  # sollte auf .venv\Scripts\python.exe enden
print("OpenCV Version:", cv2.__version__)

# Pfade relativ zum Projektordner (das Notebook liegt in notebooks/).
PROJECT_ROOT = Path.cwd().parent
INPUT_VIDEO = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
OUTPUT_VIDEO = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "VID_76_yunet_pixelated.mp4"
LOG_CSV = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "VID_76_detections.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "face_detection_yunet_2023mar.onnx"

print("Video gefunden:", INPUT_VIDEO.exists())
print("Modell gefunden:", MODEL_PATH.exists())


## Schritt 2: Parameter

- `SCORE_THRESHOLD`: Mindest-Sicherheit (0 bis 1), ab der YuNet eine Erkennung gelten lässt. Niedrigerer Wert findet mehr Gesichter, aber auch mehr Fehlalarme (z. B. Muster auf der Decke). Für Anonymisierung ist ein **verpasstes** Gesicht schlimmer als ein Fehlalarm, deshalb 0,3 statt 0,5.
- `N_BLOCKS`: Anzahl der Blöcke in der Breite des pixelierten Bereichs. **Kleiner = stärker anonymisiert.**
- `HOLD_FRAMES`: Wird in einem Frame kein Gesicht erkannt, pixelieren wir die letzte bekannte Box noch so viele Frames weiter. 15 Frames sind bei 25 fps etwa 0,6 Sekunden.
- `DETECT_SCALE`: Bei 1,0 sucht YuNet im Bild in voller Auflösung (1080p). Bei 0,5 wird das Bild vorher halbiert. Das ist schneller, aber kleine oder weit entfernte Gesichter werden eher verpasst.
- `MAX_FRAMES`: Begrenzt, wie viele Frames verarbeitet werden. Für den ersten Test 500 (ca. 20 Sekunden Video). Für das ganze Video auf `None` setzen.


In [ ]:
SCORE_THRESHOLD = 0.3   # Mindest-Sicherheit der Erkennung. Niedrig = lieber Fehlalarm als verpasstes Gesicht
N_BLOCKS = 8            # Pixelation-Stärke (kleiner = stärker anonymisiert)
HOLD_FRAMES = 15        # so viele Frames wird die letzte Box weiterverwendet, wenn nichts erkannt wird
DETECT_SCALE = 1.0      # 1.0 = volle Auflösung (1080p). Bei zu langsam auf 0.5 setzen (schneller, aber kleine Gesichter eher verpasst)
MAX_FRAMES = 500        # erster Testlauf nur mit 500 Frames (ca. 20 s). Später auf None setzen für das ganze Video


## Schritt 3: Video-Metadaten lesen

Bevor wir verarbeiten, schauen wir, was für ein Video wir vor uns haben. Dabei wird **kein Bild** angezeigt, nur Zahlen.

- `cv2.VideoCapture` öffnet die Datei. `isOpened()` prüft, ob das geklappt hat (bei einem nicht unterstützten Format würde es scheitern).
- Mit `cap.get(...)` lesen wir Breite, Höhe, Bildrate (FPS) und die Anzahl der Frames.
- Dauer in Sekunden = Frames / FPS.
- `cap.release()` gibt die Datei wieder frei. Das machen wir immer, wenn wir fertig sind.

Erwartet für dieses Video: 1920 × 1080, 25 FPS, 8759 Frames, ca. 350 Sekunden.


In [ ]:
cap = cv2.VideoCapture(str(INPUT_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f"Video konnte nicht geöffnet werden: {INPUT_VIDEO}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()  # Datei wieder freigeben

print(f"Auflösung: {width} x {height}, FPS: {fps:.2f}, Frames: {n_frames}, Dauer: {n_frames / fps:.1f} s")


## Schritt 4: Pixelation eines Bildbereichs

Diese Funktion pixeliert **eine** Box `(x, y, Breite, Höhe)` in einem Frame und gibt das veränderte Bild zurück. Das Original bleibt unverändert (wir arbeiten auf einer Kopie).

Ablauf:
1. **Box vergrößern.** YuNet-Boxen sitzen eng auf dem Gesicht. Damit Haare, Ohren und Kinn mit verdeckt werden, geben wir links, rechts und unten 30 % Rand (`padding`) und oben 60 % (`padding_top`) zu. Dann schneiden wir die Box an den Bildrändern ab, sonst gäbe es Fehler bei Gesichtern am Rand.
2. **Ausschnitt verkleinern** auf `n_blocks` Pixel Breite (`INTER_AREA`). Jeder dieser Pixel ist der Durchschnitt eines Blocks.
3. **Wieder auf Originalgröße vergrößern**, ohne zu glätten (`INTER_NEAREST`). Jeder Pixel wird so zu einem sichtbaren Block.
4. Den Ausschnitt in die Kopie des Frames zurückschreiben.

Wichtig: Die Stärke hängt über `n_blocks` nur von der Blockanzahl ab, nicht von der Gesichtsgröße. Ein großes und ein kleines Gesicht sind damit gleich stark unkenntlich.


In [ ]:
def pixelate_region(frame, box, n_blocks=8, padding=0.3, padding_top=0.6, inplace=False):
    """Pixeliert die Box (x, y, w, h) im Frame.

    padding:     Rand links/rechts/unten (Anteil von Breite bzw. Höhe)
    padding_top: größerer Rand oben, damit auch Haare/Scheitel bedeckt sind
    inplace:     True = das übergebene Bild direkt verändern (schneller, keine Kopie)
    """
    x, y, w, h = box
    img_h, img_w = frame.shape[:2]

    # Box vergrößern, danach an den Bildrändern abschneiden
    x1 = max(x - int(w * padding), 0)
    y1 = max(y - int(h * padding_top), 0)
    x2 = min(x + w + int(w * padding), img_w)
    y2 = min(y + h + int(h * padding), img_h)
    if x2 <= x1 or y2 <= y1:  # Box liegt komplett außerhalb des Bildes
        return frame if inplace else frame.copy()

    out = frame if inplace else frame.copy()
    roi = out[y1:y2, x1:x2]
    roi_h, roi_w = roi.shape[:2]

    # Verkleinern auf n_blocks Blöcke in der Breite (Höhe im gleichen Verhältnis, mind. 1)
    small_w = min(n_blocks, roi_w)
    small_h = max(1, round(roi_h * small_w / roi_w))
    small = cv2.resize(roi, (small_w, small_h), interpolation=cv2.INTER_AREA)

    # Wieder vergrößern, ohne zu glätten -> sichtbare Blöcke
    out[y1:y2, x1:x2] = cv2.resize(small, (roi_w, roi_h), interpolation=cv2.INTER_NEAREST)
    return out


## Schritt 5: Die ganze Pipeline als Funktion

`anonymize_video` fasst alles zusammen: Frame lesen, Gesichter erkennen, pixelieren, in ein neues Video schreiben. Weil es eine Funktion ist, kannst du sie später mit anderen Parametern oder anderen Videos wiederverwenden.

**Detektor erzeugen.** `cv2.FaceDetectorYN.create` lädt das YuNet-Modell. Die Bildgröße wird als `(Breite, Höhe)` angegeben. Achtung auf die Reihenfolge, sie ist anders als bei `frame.shape`. Mit `nms_threshold=0.3` werden überlappende Boxen zu einer zusammengefasst.

**Schleife über alle Frames.**
- Bei `DETECT_SCALE < 1` wird der Frame nur für die Erkennung verkleinert. Die Boxen rechnen wir danach wieder auf die volle Größe zurück (`/ detect_scale`). Pixeliert wird immer auf dem Originalframe.
- `detector.detect` liefert pro Gesicht eine Zeile mit Box, 5 Gesichtspunkten und Score. Wurde nichts gefunden, ist das Ergebnis `None`.

**Drei Zustände pro Frame**, die wir im Protokoll festhalten:
- `erkannt`: YuNet hat mindestens ein Gesicht gefunden. Wir merken uns die Boxen.
- `gehalten`: Nichts erkannt, aber vor weniger als `hold_frames` Frames gab es eine Box. Wir pixeln die alte Box weiter. Das überbrückt kurze Aussetzer, etwa bei Kopfdrehung oder Bewegungsunschärfe.
- `ungeschuetzt`: Nichts erkannt und keine alte Box mehr. Hier wird **nichts** pixeliert. Das sind die kritischen Frames.

**`try/finally`.** Egal ob ein Fehler auftritt: Video und Writer werden am Ende immer freigegeben. Sonst bleibt die Ausgabedatei unter Umständen kaputt oder gesperrt.

**Protokoll.** Pro Frame wird eine Zeile in `VID_76_detections.csv` gespeichert (Frame-Nummer, Anzahl Gesichter, bester Score, Status). Das brauchst du später, wenn du verschiedene Detektoren oder Einstellungen vergleichst.

**Rückgabe.** Die Funktion liefert eine Zusammenfassung: Anzahl Frames je Zustand, die Indizes der ungeschützten Frames, die längste Lücke ohne Schutz und die Laufzeit.

Hinweis: `mp4v` ist ein verlustbehafteter Codec, der ohne Zusatzsoftware funktioniert. Ton wird nicht übernommen, OpenCV schreibt nur Bild.


In [ ]:
def anonymize_video(input_path, output_path, model_path, score_threshold=0.3,
                    n_blocks=8, hold_frames=15, detect_scale=1.0,
                    padding=0.3, padding_top=0.6, trail_frames=5,
                    debug=False, max_frames=None, log_csv=None):
    """Pixeliert alle Gesichter im Video. Gibt eine Zusammenfassung als Dict zurück.

    trail_frames: Boxen der letzten N Frames werden mitpixeliert (fängt Bewegung ab)
    debug:        True = Boxen und Scores ins AUSGABEvideo einzeichnen (nur zur Kontrolle!)
    """
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise RuntimeError(f"Video konnte nicht geöffnet werden: {input_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    n_frames = total if max_frames is None else min(total, max_frames)

    # Der Detektor arbeitet auf der (evtl. verkleinerten) Bildgröße. Reihenfolge: (Breite, Höhe)
    det_w, det_h = int(width * detect_scale), int(height * detect_scale)
    detector = cv2.FaceDetectorYN.create(
        str(model_path), "", (det_w, det_h),
        score_threshold=score_threshold,
        nms_threshold=0.3,   # überlappende Boxen zusammenfassen
        top_k=5000,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(
        str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
    )

    rows = []              # Protokoll: eine Zeile pro Frame
    unprotected = []       # Indizes der Frames ohne jede Pixelation
    detected_frames = 0    # Frames mit frischer Erkennung
    held_frames = 0        # Frames mit weiterverwendeter alter Box
    last_boxes, age = [], 0
    recent = deque(maxlen=trail_frames + 1)  # Boxen der letzten Frames
    start = time.time()

    try:
        for i in range(n_frames):
            ok, frame = cap.read()
            if not ok:
                break

            # Zum Erkennen ggf. verkleinern (schneller), Boxen danach zurückrechnen
            if detect_scale == 1.0:
                small = frame
            else:
                small = cv2.resize(frame, (det_w, det_h), interpolation=cv2.INTER_AREA)
            _, faces = detector.detect(small)  # faces ist None, wenn nichts gefunden wurde

            if faces is not None:
                last_boxes = [(f[:4] / detect_scale).astype(int) for f in faces]
                n_faces, best_score = len(faces), float(faces[:, -1].max())
                age = 0
                detected_frames += 1
                status = "erkannt"
            else:
                n_faces, best_score = 0, 0.0
                age += 1
                if last_boxes and age <= hold_frames:
                    held_frames += 1
                    status = "gehalten"
                else:
                    last_boxes = []
                    status = "ungeschuetzt"

            # Boxen der letzten Frames mitnehmen: fängt schnelle Kopfbewegungen ab
            recent.append(list(last_boxes))
            boxes_to_blur = [b for boxes in recent for b in boxes]
            if status == "ungeschuetzt":
                if boxes_to_blur:
                    status = "nachgezogen"   # vom Nachlauf der letzten Frames noch gedeckt
                else:
                    unprotected.append(i)

            out = frame.copy()  # nur EINE Kopie pro Frame
            for box in boxes_to_blur:
                pixelate_region(out, box, n_blocks=n_blocks, padding=padding,
                                padding_top=padding_top, inplace=True)

            # Nur zur Kontrolle: erkannte Boxen mit Score ins Ausgabevideo zeichnen
            if debug and faces is not None:
                for f in faces:
                    x, y, w, h = (f[:4] / detect_scale).astype(int)
                    cv2.rectangle(out, (x, y), (x + w, y + h), (0, 255, 0), 2)
                    cv2.putText(out, f"{f[-1]:.2f}", (x, max(y - 5, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            writer.write(out)
            rows.append((i, n_faces, round(best_score, 3), status))
    finally:
        cap.release()   # egal was passiert: Dateien freigeben
        writer.release()

    # Protokoll als CSV speichern (Grundlage für die spätere Auswertung)
    if log_csv is not None:
        log_csv.parent.mkdir(parents=True, exist_ok=True)
        with open(log_csv, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["frame", "n_faces", "best_score", "status"])
            w.writerows(rows)

    # Längste Lücke ohne Schutz (aufeinanderfolgende Frame-Indizes)
    longest_gap, run = 0, 0
    for k, idx in enumerate(unprotected):
        run = run + 1 if k > 0 and idx == unprotected[k - 1] + 1 else 1
        longest_gap = max(longest_gap, run)

    return {
        "frames": len(rows),
        "erkannt": detected_frames,
        "gehalten": held_frames,
        "nachgezogen": sum(1 for r in rows if r[3] == "nachgezogen"),
        "ungeschuetzt": len(unprotected),
        "laengste_luecke": longest_gap,
        "ungeschuetzte_frames": unprotected,
        "sekunden": round(time.time() - start, 1),
    }


## Schritt 6: Pipeline ausführen

Wir rufen die Funktion mit den Parametern aus Schritt 2 auf und geben eine Zusammenfassung aus.

So liest du die Zahlen:
- **mit Erkennung:** Frames, in denen YuNet selbst ein Gesicht gefunden hat. Das sollte der Großteil sein.
- **mit gehaltener Box:** Kurze Aussetzer, die wir überbrückt haben. Viele davon deuten auf unsichere Erkennung hin.
- **ohne Schutz:** Frames, in denen nichts pixeliert wurde. **Das Ziel ist 0.** Jeder dieser Frames kann ein erkennbares Gesicht zeigen.
- **längste Lücke:** Wie viele Frames am Stück ungeschützt waren. Ein einzelner Frame ist weniger kritisch als zwei Sekunden am Stück.

Beginne mit `MAX_FRAMES = 500`. Wenn die Laufzeit und die Zahlen sinnvoll aussehen, setze `MAX_FRAMES = None` und führe Schritt 2 und 6 für das ganze Video erneut aus.


In [ ]:
DEBUG_VIDEO = PROJECT_ROOT / "data" / "processed" / "PAT_1" / "VID_76_debug.mp4"

result = anonymize_video(
    INPUT_VIDEO, DEBUG_VIDEO, MODEL_PATH,
    score_threshold=SCORE_THRESHOLD,
    n_blocks=N_BLOCKS,
    hold_frames=HOLD_FRAMES,
    detect_scale=DETECT_SCALE,
    padding=0.5,          # etwas größerer Rand als vorher (0.3)
    padding_top=0.8,
    trail_frames=5,
    debug=True,           # Boxen + Scores einzeichnen
    max_frames=MAX_FRAMES,
    log_csv=LOG_CSV,
)

print(f"Frames: {result['frames']} (Dauer: {result['sekunden']} s)")
print(f"  erkannt: {result['erkannt']} | gehalten: {result['gehalten']} | nachgezogen: {result['nachgezogen']}")
print(f"  ohne Schutz: {result['ungeschuetzt']} (längste Lücke: {result['laengste_luecke']} Frames)")


## Schritt 7: Getrennte Ansichten (Farbe / S/W)

Das Video zeigt zwei Kameras: links die **Farbansicht**, rechts oben das **S/W-Fenster** (siehe `docs/SAM3_LOG.md`, Abschnitt 2). Die bisherige Funktion (Schritt 5) sucht auf dem ganzen Bild. `anonymize_video_views` behandelt beide Ansichten getrennt:

| | `anonymize_video` (Schritt 5) | `anonymize_video_views` (neu) |
|---|---|---|
| Suche | ganzes Bild | **jede Ansicht einzeln** (eigener Detektor in passender Größe) |
| hold / trail | gemeinsam fürs ganze Bild | **pro Ansicht**: ein Treffer in Farbe löscht keine gehaltene Box in S/W |
| zu große Boxen | werden mit Rand verpixelt | **clip** auf letzte gute Gesichtsbox, sonst **full** (siehe unten) |
| außerhalb der Ansichten | bleibt sichtbar | **schwarz** |
| Start-Frame | immer ab 0 | `start` / `end` wählbar |
| Ausgabe | 1 Video + CSV | `_pixel.mp4` (Ergebnis), `_masken.mp4` (Kontrollvideo), `.pkl` (alle Boxen) |

Gleich geblieben: Score-Schwelle (Schritt 2), Verpixelung (`pixelate_region`, Schritt 4), Rand, hold und trail.

**Zu große Boxen (`MAX_FACE_FRAC`):** Im Ausschnitt meldet YuNet in S/W oft ein riesiges „Gesicht“ (Score meist 0,30–0,45, viel größer als ein echtes Gesicht). Laut Sichtprüfung steckt darin manchmal ein echtes Gesicht, aber **nicht in der Mitte**. Mit Rand und trail würde fast die ganze Ansicht verpixelt. Deshalb, wie bei SAM 3 („clip“/„full“, `docs/SAM3_LOG.md` 3.2):
- **clip:** Gab es in dieser Ansicht vor höchstens `MAX_GOOD_AGE` Frames eine normale Gesichtsbox, deren **Mittelpunkt in der großen Box liegt**, wird **diese Gesichtsbox** (mit normalem Rand) verpixelt statt der großen Box. Sie zählt wie eine Erkennung, auch für hold und trail.
- **full:** sonst wird die große Box **ganz** verpixelt (ohne Rand, nur in diesem Frame). Lieber zu viel als ein Leck.
- `MAX_FACE_FRAC = None` schaltet die Sonderbehandlung ab (große Boxen werden dann normal verpixelt).

**Kontrollvideo (`_masken.mp4`)**, eingezeichnet ist jeweils der **verpixelte Bereich**:
- **grün**: in diesem Frame erkannt (mit Score), dünn innen die eigentliche Gesichtsbox
- **türkis**: clip (letzte gute Gesichtsbox statt einer zu großen Box)
- **magenta**: full (zu große Box ganz verpixelt)
- **rot mit X, dünn**: die ursprüngliche zu große Box
- **gelb**: nicht erkannt, letzte Box gehalten (hold)
- **orange, dünn**: Box aus den letzten Frames (trail)
- **weiß**: Grenzen der Ansichten; abgedunkelt = im Ergebnis schwarz

`save_masks`: Frame-Nummern, für die die verpixelte Fläche als `data/interim/PAT_1/eval/masks/VID_76_f<Frame>_yunet.npz` gespeichert wird (für `Evaluation.ipynb`).

In [ ]:
import pickle

VIEWS = {"F": (0, 0, 961, 1080), "SW": (991, 10, 1910, 527)}   # (x0, y0, x1, y1), wie in SAM3.ipynb
PADDING, PADDING_TOP, TRAIL_FRAMES = 0.5, 0.8, 5                # wie im letzten Lauf (Schritt 6)
MAX_FACE_FRAC = 0.4    # Box breiter/höher als 40 % der Ansicht -> zu groß -> clip oder full (None = aus)
MAX_GOOD_AGE  = 50     # letzte gute Gesichtsbox für clip höchstens so viele Frames alt (50 = 2 s)

OUT_DIR     = PROJECT_ROOT / "data" / "processed" / "PAT_1"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / "PAT_1"
MASK_DIR    = INTERIM_DIR / "eval" / "masks"

# Farben im Kontrollvideo (BGR)
COLORS = {"ok": (0, 255, 0), "clip": (255, 255, 0), "full": (255, 0, 255),
          "hold": (0, 255, 255), "trail": (0, 165, 255)}
COL_RAW = (0, 0, 255)   # rot: ursprüngliche zu große Box


def put_label(img, text, org, color, scale=0.7):
    # Beschriftung mit dunklem Rand, damit sie überall lesbar ist
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)


def padded_rect(box, img_w, img_h, padding, padding_top):
    # Das Rechteck, das pixelate_region verpixelt (gleiche Rechnung), als (x1, y1, x2, y2)
    x, y, w, h = box
    return (max(x - int(w * padding), 0), max(y - int(h * padding_top), 0),
            min(x + w + int(w * padding), img_w), min(y + h + int(h * padding), img_h))


def center_inside(small, big):
    # True, wenn der Mittelpunkt der Box small (x, y, w, h) in der Box big liegt
    cx, cy = small[0] + small[2] / 2, small[1] + small[3] / 2
    return big[0] <= cx <= big[0] + big[2] and big[1] <= cy <= big[1] + big[3]


def anonymize_video_views(input_path, model_path, start=0, end=None, score_threshold=SCORE_THRESHOLD,
                          n_blocks=N_BLOCKS, hold_frames=HOLD_FRAMES, padding=PADDING, padding_top=PADDING_TOP,
                          trail_frames=TRAIL_FRAMES, max_frac=MAX_FACE_FRAC, max_good_age=MAX_GOOD_AGE,
                          save_masks=()):
    # Pixeliert Gesichter getrennt in Farb- und S/W-Ansicht. Speichert auch bei Abbruch (Stopp-Knopf).
    cap = cv2.VideoCapture(str(input_path))
    if not cap.isOpened():
        raise RuntimeError(f"Video konnte nicht geöffnet werden: {input_path}")
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps, total = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = total if end is None else min(end, total)
    for _ in range(start):              # bis zum Start vorlesen (bei AVI ist cap.set ungenau)
        cap.grab()

    # Pro Ansicht: eigener Detektor in der Größe der Ansicht und eigener Zustand
    outside = np.ones((height, width), dtype=bool)   # True = keine Ansicht -> schwarz
    detectors, state = {}, {}
    for v, (x0, y0, x1, y1) in VIEWS.items():
        outside[y0:y1, x0:x1] = False
        detectors[v] = cv2.FaceDetectorYN.create(str(model_path), "", (x1 - x0, y1 - y0),
                                                 score_threshold=score_threshold, nms_threshold=0.3, top_k=5000)
        state[v] = {"last": [], "age": 0, "recent": deque(maxlen=trail_frames + 1),   # hold / trail
                    "good": [], "good_i": -10**9}                                     # letzte gute Boxen (für clip)

    stamp = time.strftime("%Y-%m-%d_%H-%M-%S")   # Datum + Uhrzeit -> nichts wird überschrieben
    stem = f"{Path(input_path).stem}_yunetviews_{start}_{end}_{stamp}"
    print("Dateien:", stem)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer_pix = cv2.VideoWriter(str(OUT_DIR / f"{stem}_pixel.mp4"), fourcc, fps, (width, height))
    writer_vis = cv2.VideoWriter(str(OUT_DIR / f"{stem}_masken.mp4"), fourcc, fps, (width, height))

    per_frame = []
    n = end - start
    t0 = time.time()
    try:
        for i in range(n):
            ok, frame = cap.read()
            if not ok:
                break
            nr = start + i
            out = frame.copy()
            out[outside] = 0                                               # Ergebnis: schwarz
            vis = frame.copy()
            vis[outside] = (vis[outside] * 0.25).astype(np.uint8)          # Kontrollvideo: abgedunkelt
            pix_mask = np.zeros((height, width), dtype=bool)               # alles, was verpixelt wird
            status, objs, rejected = {}, [], []

            for v, (x0, y0, x1, y1) in VIEWS.items():
                vw, vh, st = x1 - x0, y1 - y0, state[v]
                out_view, vis_view = out[y0:y1, x0:x1], vis[y0:y1, x0:x1]   # Ausschnitte: Änderungen landen im Frame

                # 1. Gesichter in dieser Ansicht suchen; zu große Boxen getrennt sammeln
                _, faces = detectors[v].detect(np.ascontiguousarray(frame[y0:y1, x0:x1]))
                fresh, big = [], []
                for f in ([] if faces is None else faces):
                    b, sc = f[:4].astype(int), float(f[-1])
                    if max_frac is not None and (b[2] > max_frac * vw or b[3] > max_frac * vh):
                        big.append((b, sc))
                    else:
                        fresh.append((b, sc, "ok"))
                if fresh:                                    # normale Boxen merken (für clip)
                    st["good"], st["good_i"] = [(b, sc) for b, sc, _ in fresh], i

                # 2. Zu große Boxen: clip auf letzte gute Gesichtsbox darin, sonst full
                full = []
                for b, sc in big:
                    x, y, w, h = b
                    cv2.rectangle(vis_view, (x, y), (x + w, y + h), COL_RAW, 1)          # rot mit X
                    cv2.line(vis_view, (x, y), (x + w, y + h), COL_RAW, 1)
                    cv2.line(vis_view, (x, y + h), (x + w, y), COL_RAW, 1)
                    inside = []
                    if i - st["good_i"] <= max_good_age:
                        inside = [gb for gb, _ in st["good"] if center_inside(gb, b)]
                    if inside:
                        mode = "clip"
                        if st["good_i"] != i:                # in diesem Frame schon normal erkannt -> nichts extra
                            fresh += [(gb, sc, "clip") for gb in inside]
                    else:
                        mode = "full"
                        full.append((b, sc))
                    rejected.append({"view": v, "score": sc, "mode": mode,
                                     "box": [int(x + x0), int(y + y0), int(x + w + x0), int(y + h + y0)]})

                # 3. hold: ohne frischen Treffer die letzten Boxen bis hold_frames weiterverwenden
                if fresh:
                    st["last"], st["age"] = fresh, 0
                    s = "erkannt" if any(m == "ok" for _, _, m in fresh) else "clip"
                else:
                    st["age"] += 1
                    if st["last"] and st["age"] <= hold_frames:
                        s = "gehalten"
                    else:
                        st["last"], s = [], "ungeschuetzt"

                # 4. trail: Boxen der letzten Frames zusätzlich verpixeln
                st["recent"].append(list(st["last"]))
                current = [(b, sc, m if s in ("erkannt", "clip") else "hold") for b, sc, m in st["last"]]
                trail = [(b, sc, "trail") for bs in list(st["recent"])[:-1] for b, sc, _ in bs]
                if s == "ungeschuetzt" and trail:
                    s = "nachgezogen"      # nur noch vom trail gedeckt
                if s == "ungeschuetzt" and full:
                    s = "full"             # nur noch die ganz verpixelte große Box
                status[v] = s

                # 5. Verpixeln (Ergebnis) und einzeichnen (Kontrollvideo); aktuelle Boxen zuletzt -> oben
                todo = [(b, sc, "full", 0.0, 0.0) for b, sc in full] + \
                       [(b, sc, m, padding, padding_top) for b, sc, m in trail + current]
                for b, sc, mode, pad, pad_top in todo:
                    pixelate_region(out_view, b, n_blocks=n_blocks, padding=pad, padding_top=pad_top, inplace=True)
                    rx1, ry1, rx2, ry2 = padded_rect(b, vw, vh, pad, pad_top)
                    pix_mask[y0 + ry1:y0 + ry2, x0 + rx1:x0 + rx2] = True
                    color = COLORS[mode]
                    cv2.rectangle(vis_view, (rx1, ry1), (rx2 - 1, ry2 - 1), color, 1 if mode == "trail" else 2)
                    if mode != "trail":
                        x, y, w, h = b
                        if mode != "full":
                            cv2.rectangle(vis_view, (x, y), (x + w, y + h), color, 1)
                        put_label(vis, f"{v} {sc:.2f}" + ("" if mode == "ok" else f" {mode}"),
                                  (x0 + rx1, max(y0 + ry1 - 8, 26)), color)
                        objs.append({"view": v, "score": sc, "mode": mode,
                                     "box": [int(x + x0), int(y + y0), int(x + w + x0), int(y + h + y0)]})

            for x0, y0, x1, y1 in VIEWS.values():     # Grenzen der Ansichten (zur Kontrolle)
                cv2.rectangle(vis, (x0, y0), (x1 - 1, y1 - 1), (255, 255, 255), 1)
            put_label(vis, f"Frame {nr}", (20, 40), (255, 255, 255), scale=1.0)
            writer_pix.write(out)
            writer_vis.write(vis)
            per_frame.append({"frame": nr, "status": status, "objs": objs, "rejected": rejected})

            if nr in save_masks:   # für Evaluation.ipynb: genau die Fläche, die verpixelt wurde
                MASK_DIR.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(MASK_DIR / f"{Path(input_path).stem}_f{nr:05d}_yunet.npz", mask=pix_mask,
                                    run=stem, score_threshold=score_threshold, hold=hold_frames, trail=trail_frames,
                                    padding=padding, padding_top=padding_top, max_good_age=max_good_age,
                                    max_frac=-1.0 if max_frac is None else max_frac)
                print(f"Maske für Frame {nr} gespeichert ({int(pix_mask.sum())} px verpixelt)")

            if i % 500 == 0:
                print(f"Frame {nr:5d} ({i}/{n})  {status}  {time.time() - t0:6.1f} s")
    finally:
        # Läuft immer, auch beim Stopp-Knopf -> Videos abspielbar, Ergebnisse gespeichert
        cap.release()
        writer_pix.release()
        writer_vis.release()
        el = time.time() - t0
        print(f"Ende: {len(per_frame)} Frames in {el:.1f} s ({len(per_frame) / max(el, 1e-9):.1f} Frames/s)")
        with open(INTERIM_DIR / f"{stem}.pkl", "wb") as f:
            pickle.dump(per_frame, f)
    return per_frame

In [ ]:
def longest_run(flags):
    # Längste Folge von True hintereinander
    longest, cur = 0, 0
    for f in flags:
        cur = cur + 1 if f else 0
        longest = max(longest, cur)
    return longest


def gap_stats_views(per_frame, name):
    # Pro Ansicht:
    # - ohne Verpixelung: in dieser Ansicht wurde gar nichts verpixelt (Ziel 0)
    # - ohne frische Erkennung: kein normales Gesicht gefunden; dann greifen clip, hold, trail oder full
    # - Status-Zählung: erkannt / clip / gehalten / nur trail / nur full / ungeschützt
    # - zu groß: Frames mit mindestens einer zu großen Box, davon mit clip bzw. full behandelt
    # - Gesichter pro Frame: normale Erkennungen (mehr als Personen im Bild = Fehlerkennungen)
    for v, label in [("F", "Farbe"), ("SW", "S/W")]:
        sts = [f["status"][v] for f in per_frame]
        none = [s == "ungeschuetzt" for s in sts]
        no_fresh = [s != "erkannt" for s in sts]
        rej = [[r for r in f["rejected"] if r["view"] == v] for f in per_frame]
        n_big = sum(bool(r) for r in rej)
        n_clip = sum(any(x["mode"] == "clip" for x in r) for r in rej)
        n_full = sum(any(x["mode"] == "full" for x in r) for r in rej)
        per = [sum(o["view"] == v and o["mode"] == "ok" for o in f["objs"]) for f in per_frame]
        dist = {k: per.count(k) for k in sorted(set(per))}
        sc = [o["score"] for f in per_frame for o in f["objs"] if o["view"] == v and o["mode"] == "ok"]
        sc_txt = np.percentile(sc, [0, 50, 100]).round(2) if sc else "-"
        counts = {k: sts.count(k) for k in ["erkannt", "clip", "gehalten", "nachgezogen", "full", "ungeschuetzt"]}
        print(f"{name} {label}:")
        print(f"   ohne Verpixelung {sum(none)} (längste Lücke {longest_run(none)}) | "
              f"ohne frische Erkennung {sum(no_fresh)} (längste {longest_run(no_fresh)})")
        print(f"   Status {counts}")
        print(f"   zu groß in {n_big} Frames (davon clip {n_clip}, full {n_full}) | "
              f"Gesichter pro Frame {dist} | Score min/median/max {sc_txt}")

## Schritt 8: Läufe mit getrennten Ansichten

Ergebnisse: `data/processed/PAT_1/VID_76_yunetviews_<start>_<end>_<Datum>_pixel.mp4` und `..._masken.mp4`, Boxen in `data/interim/PAT_1/...pkl`.

1. **Test** auf 2400–3000 (dieselbe Stelle wie bei SAM 3, ca. 1–2 min). Im Kontrollvideo prüfen: Sitzen die weißen Rahmen auf den Ansichten? Was steckt hinter roten X-Boxen (verworfen) und gelben Boxen (hold)?
2. **Ganzes Video** (8759 Frames, ca. 15–25 min).
3. Gespeicherten Lauf erneut auswerten, ohne neu zu rechnen.

In [ ]:
# 1. Test auf der schwierigen Stelle
res_test = anonymize_video_views(INPUT_VIDEO, MODEL_PATH, start=2400, end=3000)
gap_stats_views(res_test, "2400–3000")

In [ ]:
# 2. Ganzes Video (bricht man ab, werden Videos und .pkl trotzdem sauber gespeichert)
res_full = anonymize_video_views(INPUT_VIDEO, MODEL_PATH)
gap_stats_views(res_full, "ganzes Video")

In [ ]:
# 3. Neuesten gespeicherten Lauf über das ganze Video auswerten, ohne neu zu rechnen
runs = sorted(INTERIM_DIR.glob(f"{INPUT_VIDEO.stem}_yunetviews_0_*.pkl"))
if runs:
    print("Lauf:", runs[-1].name)
    gap_stats_views(pickle.load(open(runs[-1], "rb")), "ganzes Video")
else:
    print("noch kein Lauf über das ganze Video")

### Masken für die Evaluation (markierte Frames)

Für jeden in X-AnyLabeling markierten Frame (`data/interim/PAT_1/gt/*.json`) läuft die Pipeline ab `LEAD` Frames vorher, damit hold und trail im gleichen Zustand sind wie im ganzen Lauf. Die Maske landet in `data/interim/PAT_1/eval/masks/` und wird von `Evaluation.ipynb` gelesen.

In [ ]:
GT_FRAMES = sorted(int(p.stem.split("_f")[-1]) for p in (INTERIM_DIR / "gt").glob(f"{INPUT_VIDEO.stem}_f*.json"))
LEAD = 50   # 2 s Vorlauf (hold 15 + trail 5 Frames reichen weniger weit zurück)
print("Markierte Frames:", GT_FRAMES)

for f in GT_FRAMES:
    anonymize_video_views(INPUT_VIDEO, MODEL_PATH, start=max(0, f - LEAD), end=f + 1, save_masks={f})